# 1. Raw Data Exploration

Explore the M5 dataset structure and statistics.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_calendar, load_prices, load_sales

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Raw Data

In [ ]:
raw_path = './data/raw'

print("Loading raw M5 data...")
calendar = load_calendar(raw_path)
prices = load_prices(raw_path)
sales = load_sales(raw_path)

print(f"Calendar shape: {calendar.shape}")
print(f"Prices shape: {prices.shape}")
print(f"Sales shape: {sales.shape}")

## Explore Calendar Data

In [ ]:
print("Calendar Data:")
print(calendar.head())
print(f"\nColumns: {calendar.columns.tolist()}")
print(f"\nDate range: {calendar['date'].min()} to {calendar['date'].max()}")
print(f"\nEvent types:")
print(calendar['event_type_1'].value_counts().head(10))

## Explore Prices Data

In [ ]:
print("Prices Data:")
print(prices.head())
print(f"\nStores: {prices['store_id'].nunique()}")
print(f"Items: {prices['item_id'].nunique()}")
print(f"\nPrice statistics:")
print(prices['sell_price'].describe())

## Explore Sales Data

In [ ]:
print("Sales Data (first 5 rows, first 10 columns):")
print(sales.iloc[:5, :10])
print(f"\nStores: {sales['store_id'].nunique()}")
print(f"Items: {sales['item_id'].nunique()}")
print(f"Categories: {sales['cat_id'].nunique()}")
print(f"Departments: {sales['dept_id'].nunique()}")

# Get sales columns (d_1, d_2, ...)
sales_cols = [col for col in sales.columns if col.startswith('d_')]
print(f"\nNumber of time periods: {len(sales_cols)}")

## Sales Distribution

In [ ]:
# Calculate average sales per day
sales_cols = [col for col in sales.columns if col.startswith('d_')]
daily_sales = sales[sales_cols].mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Time series of average sales
axes[0].plot(daily_sales.values)
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales Over Time')
axes[0].grid(True, alpha=0.3)

# Distribution of average item sales
item_avg_sales = sales[sales_cols].mean(axis=1)
axes[1].hist(item_avg_sales, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Average Sales')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Item Average Sales')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Average daily sales: {daily_sales.mean():.2f}")
print(f"Average item sales: {item_avg_sales.mean():.2f}")

## Store Analysis

In [ ]:
store_stats = sales.groupby('store_id')[sales_cols].mean().mean(axis=1).sort_values(ascending=False)

print("Average Sales by Store:")
print(store_stats)

plt.figure(figsize=(10, 6))
store_stats.plot(kind='barh')
plt.xlabel('Average Daily Sales')
plt.title('Store Performance')
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("\n=== DATASET SUMMARY ===")
print(f"Time periods: {len(sales_cols)} days")
print(f"Stores: {sales['store_id'].nunique()}")
print(f"Categories: {sales['cat_id'].nunique()}")
print(f"Items: {sales['item_id'].nunique()}")
print(f"\nTotal items (store-item combinations): {len(sales)}")
print(f"\nCalendar features:")
print(f"  - Dates: {calendar['date'].nunique()}")
print(f"  - Events tracked: {calendar['event_type_1'].nunique()} unique event types")